In [ ]:
!pip -q install pandas==2.2.2 "scikit-learn>=1.2,<1.9"
!pip -q install requests python-docx jinja2 statsmodels openai

import os
import shutil
import pandas as pd
from google.colab import files

print("pandas:", pd.__version__)

In [ ]:
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

PROJECT_DIR = "/content/qckur_project"

ds_file = Path(PROJECT_DIR) / "datasciencecomponents.py"

text = ds_file.read_text(encoding="utf-8")

if "from pycaret import classification,regression" in text:
    text = text.replace(
        "from pycaret import classification,regression",
        "try:\n"
        "    from pycaret import classification, regression\n"
        "except Exception:\n"
        "    classification = None\n"
        "    regression = None"
    )

ds_file.write_text(text, encoding="utf-8")

print("Patched datasciencecomponents.py")


In [ ]:
# =========================================================
# 1. API key and paths
# =========================================================

import os
import sys
import json
import time
import re
import requests
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

DEEPSEEK_API_KEY = "key"
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY

LLM_PROVIDER = "DeepSeek"
LLM_MODEL_ID = "deepseek-v4-pro"
LLM_DISPLAY_NAME = "DeepSeek-V4-Pro"

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_CHAT_COMPLETIONS_URL = f"{DEEPSEEK_BASE_URL}/chat/completions"

DEEPSEEK_THINKING_TYPE = "disabled"

PROJECT_DIR = "/content/qckur_project"
CONFIG_PATH = "/content/qckur_project/configs/regression_linear_regression_experiment_config.json"
DATA_DIR = "/content/qckur_project/data"
OUTPUT_DIR = "/content/qckur_project/prototype_linear_regression_outputs_deepseek_v4"

OPENAI_MODEL = LLM_MODEL_ID

MATCHING_TEMPERATURE = 0.0
REFINEMENT_TEMPERATURE = 0.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.append(PROJECT_DIR)


# =========================================================
# 2. Patch optional pycaret import in NLGcomponents.py
# =========================================================

nlg_file = Path(PROJECT_DIR) / "NLGcomponents.py"

if nlg_file.exists():
    text = nlg_file.read_text(encoding="utf-8")
    if "from pycaret import classification,regression" in text:
        text = text.replace(
            "from pycaret import classification,regression",
            "try:\n"
            "    from pycaret import classification, regression\n"
            "except Exception:\n"
            "    classification = None\n"
            "    regression = None"
        )
        nlg_file.write_text(text, encoding="utf-8")


# =========================================================
# 3. Imports
# =========================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifier
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import r2_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from NLGcomponents import (
    RegressionTemplateBasedTextGeneration,
    ClassifierTemplateBasedTextGeneration,
    SettingForChatGPT,
    LoadQuestionBank,
)


# =========================================================
# 4. Initialize prototype components
# =========================================================

set_for_GPT = SettingForChatGPT()
nlg_reg = RegressionTemplateBasedTextGeneration()
nlg_cls = ClassifierTemplateBasedTextGeneration()
loader = LoadQuestionBank()


# =========================================================
# 5. Config and file helpers
# =========================================================

def safe_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(name).strip())


def load_configs(config_path: str) -> List[Dict[str, Any]]:
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        configs = json.load(f)

    if not isinstance(configs, list):
        raise ValueError("Config JSON must contain a list of experiment configs.")

    return configs


def get_dataset_path(config: Dict[str, Any]) -> str:
    if config.get("dataset_path"):
        return config["dataset_path"]
    return os.path.join(DATA_DIR, config["file_name"])


def load_dataset(config: Dict[str, Any]) -> pd.DataFrame:
    sep = config.get("csv_sep", ",")
    dataset_path = get_dataset_path(config)

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    return pd.read_csv(dataset_path, sep=sep)


def get_model_name(config: Dict[str, Any]) -> str:
    model_name = (
        config.get("configured_model_name")
        or config.get("model_name")
        or config.get("configured_model")
    )

    if not model_name:
        raise ValueError(f"Missing model name in config: {config.get('dataset_name')}")

    return model_name


def infer_task_type(config: Dict[str, Any]) -> str:
    if config.get("task_type"):
        return config["task_type"]

    model_name = get_model_name(config)

    if model_name in [
        "linear_regression",
        "random_forest_regressor",
        "gradient_boosting_regressor",
    ]:
        return "regression"

    if model_name in [
        "logistic_regression",
        "lda_classifier",
        "ridge_classifier",
    ]:
        return "binary_classification"

    if model_name in [
        "decision_tree_classifier",
        "random_forest_classifier",
    ]:
        return "multiclass_classification"

    raise ValueError(f"Cannot infer task type from model: {model_name}")


MODEL_DISPLAY_NAMES = {
    "linear_regression": "Scikit-learn Linear Regression",
    "random_forest_regressor": "Random Forest Regression",
    "gradient_boosting_regressor": "Gradient Boosting Regression",
    "logistic_regression": "Logistic Regression",
    "lda_classifier": "Linear Discriminant Analysis",
    "ridge_classifier": "Ridge Classifier",
    "decision_tree_classifier": "Decision Tree Classifier",
    "random_forest_classifier": "Random Forest Classifier",
}


# =========================================================
# 6. JSON helpers
# =========================================================

def sanitize_for_json(obj: Any) -> Any:
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")

    if isinstance(obj, pd.Series):
        return obj.to_dict()

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, dict):
        clean = {}
        for k, v in obj.items():
            if k == "model":
                continue
            if k == "pipeline":
                continue
            clean[str(k)] = sanitize_for_json(v)
        return clean

    if isinstance(obj, list):
        return [sanitize_for_json(x) for x in obj]

    try:
        json.dumps(obj)
        return obj
    except Exception:
        return str(obj)


def extract_first_integer(text: str) -> int:
    match = re.search(r"\d+", str(text))
    if match:
        return int(match.group())
    return 0


# =========================================================
# 7. Unified preprocessing and model backend
# =========================================================

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def split_feature_types(df: pd.DataFrame, x_columns: List[str]) -> Dict[str, List[str]]:
    numeric_columns = []
    categorical_columns = []

    for col in x_columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_columns.append(col)
        else:
            categorical_columns.append(col)

    return {
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
    }


def build_unified_preprocessor(
    X: pd.DataFrame,
    x_columns: List[str],
) -> Tuple[ColumnTransformer, Dict[str, List[str]]]:
    feature_types = split_feature_types(X, x_columns)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, feature_types["numeric_columns"]),
            ("cat", categorical_transformer, feature_types["categorical_columns"]),
        ],
        remainder="drop",
    )

    return preprocessor, feature_types


def clean_preprocessed_feature_name(name: str) -> str:
    name = str(name)

    if name.startswith("num__"):
        return name.replace("num__", "", 1)

    if name.startswith("cat__"):
        return name.replace("cat__", "", 1)

    return name


def get_feature_names_from_pipeline(pipeline: Pipeline) -> List[str]:
    preprocessor = pipeline.named_steps["preprocessor"]
    raw_feature_names = preprocessor.get_feature_names_out()

    return [
        clean_preprocessed_feature_name(name)
        for name in raw_feature_names
    ]


def detect_numeric_standardization(pipeline: Pipeline) -> bool:
    """
    Detect whether the fitted sklearn pipeline standardizes numeric predictors.

    This is used to decide whether linear-regression coefficients should be
    interpreted as effects of one-standard-deviation increases rather than
    one-unit increases on the original raw scale.
    """
    try:
        preprocessor = pipeline.named_steps["preprocessor"]

        for name, transformer, columns in preprocessor.transformers_:
            if name == "num":
                if hasattr(transformer, "named_steps"):
                    scaler = transformer.named_steps.get("scaler")
                    return isinstance(scaler, StandardScaler)

        return False

    except Exception:
        return False


def build_feature_interpretation_metadata(
    feature_names: List[str],
    feature_types: Dict[str, List[str]],
    numeric_standardized: bool,
) -> Tuple[List[str], List[str]]:
    """
    Build per-feature interpretation metadata for coefficient templates.

    Numeric predictors:
    - if standardized: one-standard-deviation increase after preprocessing
    - otherwise: one-unit increase on the original scale

    One-hot encoded categorical predictors:
    - change in the encoded indicator from 0 to 1
    """
    numeric_columns = set(feature_types.get("numeric_columns", []))

    feature_type_labels = []
    interpretation_units = []

    for feature_name in feature_names:
        if feature_name in numeric_columns:
            feature_type_labels.append("numeric")

            if numeric_standardized:
                interpretation_units.append(
                    "one-standard-deviation increase after preprocessing"
                )
            else:
                interpretation_units.append(
                    "one-unit increase on the original scale"
                )

        else:
            feature_type_labels.append("categorical_or_onehot")
            interpretation_units.append(
                "a change in the one-hot encoded category indicator from 0 to 1"
            )

    return feature_type_labels, interpretation_units


def build_model(model_name: str, task_type: str):
    if task_type == "regression":
        if model_name == "linear_regression":
            return LinearRegression()

        if model_name == "random_forest_regressor":
            return RandomForestRegressor(random_state=42)

        if model_name == "gradient_boosting_regressor":
            return GradientBoostingRegressor(random_state=42)

    if task_type in ["binary_classification", "multiclass_classification"]:
        if model_name == "logistic_regression":
            return LogisticRegression(max_iter=2000, random_state=42)

        if model_name == "lda_classifier":
            return LinearDiscriminantAnalysis()

        if model_name == "ridge_classifier":
            return RidgeClassifier()

        if model_name == "decision_tree_classifier":
            return DecisionTreeClassifier(random_state=42)

        if model_name == "random_forest_classifier":
            return RandomForestClassifier(random_state=42)

    raise ValueError(f"Unsupported model '{model_name}' for task type '{task_type}'.")


def build_unified_pipeline(
    X: pd.DataFrame,
    x_columns: List[str],
    model_name: str,
    task_type: str,
) -> Tuple[Pipeline, Dict[str, List[str]]]:
    preprocessor, feature_types = build_unified_preprocessor(X, x_columns)
    model = build_model(model_name, task_type)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return pipeline, feature_types


def safe_train_test_split_for_classification(X, y):
    try:
        counts = y.value_counts(dropna=False)
        if len(counts) > 1 and counts.min() >= 2:
            return train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
                stratify=y,
            )
    except Exception:
        pass

    return train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=None,
    )


def compute_train_test_metrics(
    base_pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    task_type: str,
) -> Dict[str, Optional[float]]:
    if len(X) < 5:
        return {
            "train_metric": None,
            "test_metric": None,
            "metric_name": None,
            "split_note": "Too few rows for a stable train/test split.",
        }

    try:
        if task_type == "regression":
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
            )

            split_pipeline = clone(base_pipeline)
            split_pipeline.fit(X_train, y_train)

            train_pred = split_pipeline.predict(X_train)
            test_pred = split_pipeline.predict(X_test)

            return {
                "train_metric": float(r2_score(y_train, train_pred)),
                "test_metric": float(r2_score(y_test, test_pred)),
                "metric_name": "r_squared",
                "split_note": "Train/test split uses test_size=0.2 and random_state=42.",
            }

        X_train, X_test, y_train, y_test = safe_train_test_split_for_classification(X, y)

        split_pipeline = clone(base_pipeline)
        split_pipeline.fit(X_train, y_train)

        train_pred = split_pipeline.predict(X_train)
        test_pred = split_pipeline.predict(X_test)

        return {
            "train_metric": float(accuracy_score(y_train, train_pred)),
            "test_metric": float(accuracy_score(y_test, test_pred)),
            "metric_name": "accuracy",
            "split_note": "Train/test split uses test_size=0.2 and random_state=42; stratification is used when feasible.",
        }

    except Exception as exc:
        return {
            "train_metric": None,
            "test_metric": None,
            "metric_name": None,
            "split_note": f"Train/test metric computation failed: {str(exc)}",
        }


# =========================================================
# 8. Evidence adapters for NLG templates
# =========================================================

def try_compute_linear_regression_pvalues(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    feature_names: List[str],
) -> List[float]:
    try:
        import statsmodels.api as sm

        X_preprocessed = pipeline.named_steps["preprocessor"].transform(X)

        if hasattr(X_preprocessed, "toarray"):
            X_preprocessed = X_preprocessed.toarray()

        X_with_const = sm.add_constant(X_preprocessed, has_constant="add")
        model = sm.OLS(y.astype(float).to_numpy(), X_with_const).fit()

        pvalues = list(model.pvalues)

        if len(pvalues) == len(feature_names) + 1:
            return [float(p) for p in pvalues[1:]]

        if len(pvalues) >= len(feature_names):
            return [float(p) for p in pvalues[-len(feature_names):]]

    except Exception:
        pass

    return [1.0 for _ in feature_names]


def build_regression_evidence(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    config: Dict[str, Any],
    task_type: str,
    model_name: str,
    feature_types: Dict[str, List[str]],
) -> Dict[str, Any]:
    y_pred = pipeline.predict(X)
    r2 = float(r2_score(y, y_pred))

    feature_names = get_feature_names_from_pipeline(pipeline)
    trained_model = pipeline.named_steps["model"]

    numeric_standardized = detect_numeric_standardization(pipeline)

    feature_type_labels, interpretation_units = build_feature_interpretation_metadata(
        feature_names=feature_names,
        feature_types=feature_types,
        numeric_standardized=numeric_standardized,
    )

    if model_name == "linear_regression":
        coef_values = np.asarray(trained_model.coef_, dtype=float).ravel()
        p_values = try_compute_linear_regression_pvalues(
            pipeline=pipeline,
            X=X,
            y=y,
            feature_names=feature_names,
        )
        evidence_type = "coefficients"

    elif hasattr(trained_model, "feature_importances_"):
        coef_values = np.asarray(trained_model.feature_importances_, dtype=float).ravel()
        p_values = [1.0 for _ in feature_names]
        evidence_type = "feature_importances"

    else:
        coef_values = np.zeros(len(feature_names), dtype=float)
        p_values = [1.0 for _ in feature_names]
        evidence_type = "unavailable"

    if len(coef_values) < len(feature_names):
        coef_values = np.pad(coef_values, (0, len(feature_names) - len(coef_values)))
    elif len(coef_values) > len(feature_names):
        coef_values = coef_values[:len(feature_names)]

    if len(p_values) < len(feature_names):
        p_values = p_values + [1.0 for _ in range(len(feature_names) - len(p_values))]
    elif len(p_values) > len(feature_names):
        p_values = p_values[:len(feature_names)]

    coeff_df = pd.DataFrame(
        {
            "Xcol": feature_names,
            "Coefficients": [float(v) for v in coef_values],
            "P-values": [float(v) for v in p_values],
            "FeatureType": feature_type_labels,
            "InterpretationUnit": interpretation_units,
        }
    )

    coeff_for_template = [0.0] + [float(v) for v in coef_values]
    p_values_for_template = [1.0] + [float(v) for v in p_values]

    split_metrics = compute_train_test_metrics(
        base_pipeline=pipeline,
        X=X,
        y=y,
        task_type="regression",
    )

    if numeric_standardized:
        numeric_coefficient_note = (
            "Numeric predictors are median-imputed and standardized using StandardScaler before model fitting. "
            "Therefore, coefficients for numeric predictors should be interpreted as effects of one-standard-deviation "
            "increases after preprocessing, not as one-unit increases in the original raw variables. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded; their coefficients refer to changes "
            "in the encoded category indicators."
        )
        coefficient_interpretation_mode = "standardized_numeric_predictors"
    else:
        numeric_coefficient_note = (
            "Numeric predictors are not standardized before model fitting. "
            "Therefore, coefficients for numeric predictors can be interpreted as effects of one-unit increases "
            "on the original scale, holding other variables fixed. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded; their coefficients refer to changes "
            "in the encoded category indicators."
        )
        coefficient_interpretation_mode = "raw_numeric_predictors"

    results = {
        "selected_model": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "task_type": task_type,
        "model_name": model_name,
        "model": trained_model,
        "pipeline": pipeline,
        "coeff_df": coeff_df,
        "r_squared": r2,
        "coeff": coeff_for_template,
        "p_values": p_values_for_template,
        "train_r2": split_metrics["train_metric"],
        "test_r2": split_metrics["test_metric"],
        "numeric_columns": feature_types["numeric_columns"],
        "categorical_columns": feature_types["categorical_columns"],
        "preprocessed_feature_names": feature_names,
        "evidence_type": evidence_type,
        "numeric_standardized": numeric_standardized,
        "coefficient_interpretation_mode": coefficient_interpretation_mode,
        "coefficient_interpretation_note": numeric_coefficient_note,
        "metric_note": "The reported R-squared is calculated on the same data used for fitting.",
        "split_note": split_metrics["split_note"],
        "preprocessing_mode": numeric_coefficient_note,
    }

    return results


def build_classifier_coeff_df_and_matrix(
    trained_model: Any,
    model_name: str,
    feature_names: List[str],
    y: pd.Series,
) -> Tuple[pd.DataFrame, np.ndarray]:
    classes = list(getattr(trained_model, "classes_", sorted(pd.Series(y).dropna().unique())))

    if hasattr(trained_model, "coef_"):
        coef_matrix = np.asarray(trained_model.coef_, dtype=float)

        if coef_matrix.ndim == 1:
            coef_matrix = coef_matrix.reshape(1, -1)

        if coef_matrix.shape[0] == 1:
            row_labels = [str(classes[-1]) if classes else "positive_class"]
        else:
            row_labels = [str(c) for c in classes[:coef_matrix.shape[0]]]

    elif hasattr(trained_model, "feature_importances_"):
        importances = np.asarray(trained_model.feature_importances_, dtype=float).ravel()

        if len(importances) < len(feature_names):
            importances = np.pad(importances, (0, len(feature_names) - len(importances)))
        elif len(importances) > len(feature_names):
            importances = importances[:len(feature_names)]

        if len(classes) > 2:
            coef_matrix = np.vstack([importances for _ in classes])
            row_labels = [str(c) for c in classes]
        else:
            coef_matrix = importances.reshape(1, -1)
            row_labels = [str(classes[-1]) if classes else "positive_class"]

    else:
        coef_matrix = np.zeros((1, len(feature_names)), dtype=float)
        row_labels = [str(classes[-1]) if classes else "positive_class"]

    if coef_matrix.shape[1] < len(feature_names):
        pad_width = len(feature_names) - coef_matrix.shape[1]
        coef_matrix = np.pad(coef_matrix, ((0, 0), (0, pad_width)))
    elif coef_matrix.shape[1] > len(feature_names):
        coef_matrix = coef_matrix[:, :len(feature_names)]

    data = {
        "Intercept": [0.0 for _ in row_labels]
    }

    for idx, feature_name in enumerate(feature_names):
        data[feature_name] = [float(v) for v in coef_matrix[:, idx]]

    coeff_df = pd.DataFrame(data, index=row_labels)

    return coeff_df, coef_matrix


def build_classifier_evidence(
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    config: Dict[str, Any],
    task_type: str,
    model_name: str,
    feature_types: Dict[str, List[str]],
) -> Dict[str, Any]:
    y_pred = pipeline.predict(X)
    acc = float(accuracy_score(y, y_pred))

    feature_names = get_feature_names_from_pipeline(pipeline)
    trained_model = pipeline.named_steps["model"]

    coeff_df, coeff_matrix = build_classifier_coeff_df_and_matrix(
        trained_model=trained_model,
        model_name=model_name,
        feature_names=feature_names,
        y=y,
    )

    split_metrics = compute_train_test_metrics(
        base_pipeline=pipeline,
        X=X,
        y=y,
        task_type=task_type,
    )

    classes = list(getattr(trained_model, "classes_", sorted(pd.Series(y).dropna().unique())))
    class_distribution = y.value_counts(dropna=False).to_dict()
    class_distribution = {
        str(k): int(v)
        for k, v in class_distribution.items()
    }

    results = {
        "selected_model": MODEL_DISPLAY_NAMES.get(model_name, model_name),
        "task_type": task_type,
        "model_name": model_name,
        "model": trained_model,
        "pipeline": pipeline,
        "coeff_df": coeff_df,
        "accuracy": acc,
        "coeff": coeff_matrix,
        "train_accuracy": split_metrics["train_metric"],
        "test_accuracy": split_metrics["test_metric"],
        "numeric_columns": feature_types["numeric_columns"],
        "categorical_columns": feature_types["categorical_columns"],
        "preprocessed_feature_names": feature_names,
        "classes": [str(c) for c in classes],
        "class_distribution": class_distribution,
        "metric_note": "The reported accuracy is calculated on the same data used for fitting.",
        "split_note": split_metrics["split_note"],
        "preprocessing_mode": (
            "Numeric predictors are median-imputed and standardized. "
            "Categorical predictors are most-frequent-imputed and one-hot encoded."
        ),
    }

    return results


# =========================================================
# 9. Fit configured model with unified preprocessing
# =========================================================

def fit_model_with_unified_backend(config: Dict[str, Any]) -> Dict[str, Any]:
    df = load_dataset(config)

    x_columns = config["x_columns"]
    y_column = config["y_column"]
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    missing = [c for c in x_columns + [y_column] if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{config['dataset_name']}] Missing columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    df = df[x_columns + [y_column]].copy()
    df = df.dropna(subset=[y_column])

    X = df[x_columns].copy()
    y = df[y_column].copy()

    if task_type == "regression":
        y_numeric = pd.to_numeric(y, errors="coerce")
        valid_mask = y_numeric.notna()
        X = X.loc[valid_mask].copy()
        y = y_numeric.loc[valid_mask].copy()

    pipeline, feature_types = build_unified_pipeline(
        X=X,
        x_columns=x_columns,
        model_name=model_name,
        task_type=task_type,
    )

    pipeline.fit(X, y)

    if task_type == "regression":
        results = build_regression_evidence(
            pipeline=pipeline,
            X=X,
            y=y,
            config=config,
            task_type=task_type,
            model_name=model_name,
            feature_types=feature_types,
        )

    else:
        results = build_classifier_evidence(
            pipeline=pipeline,
            X=X,
            y=y,
            config=config,
            task_type=task_type,
            model_name=model_name,
            feature_types=feature_types,
        )

    results.update(
        {
            "dataset_name": config["dataset_name"],
            "file_name": config.get("file_name"),
            "x_columns": x_columns,
            "y_column": y_column,
            "shape_after_dropping_missing_y": list(df.shape),
            "analysis_preprocessing_mode": "unified_sklearn_pipeline_simpleimputer_standardscaler_onehotencoder",
        }
    )

    return results


# =========================================================
# 10. DeepSeek calls with token/runtime logging
# =========================================================

def normalize_usage(usage: Optional[Dict[str, Any]]) -> Dict[str, int]:
    if not usage:
        return {
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
        }

    prompt_tokens = int(
        usage.get("prompt_tokens")
        or usage.get("input_tokens")
        or usage.get("input_token_count")
        or 0
    )

    completion_tokens = int(
        usage.get("completion_tokens")
        or usage.get("output_tokens")
        or usage.get("output_token_count")
        or 0
    )

    total_tokens = int(
        usage.get("total_tokens")
        or usage.get("total_token_count")
        or prompt_tokens + completion_tokens
        or 0
    )

    return {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
    }


def summarize_llm_logs(logs: List[Dict[str, Any]]) -> Dict[str, Any]:
    prompt_tokens = 0
    completion_tokens = 0
    total_tokens = 0
    runtime_seconds = 0.0
    success_count = 0
    failure_count = 0

    for log in logs:
        usage = log.get("usage", {}) or {}

        prompt_tokens += usage.get("prompt_tokens", 0) or 0
        completion_tokens += usage.get("completion_tokens", 0) or 0
        total_tokens += usage.get("total_tokens", 0) or 0
        runtime_seconds += log.get("runtime_seconds", 0.0) or 0.0

        if log.get("success"):
            success_count += 1
        else:
            failure_count += 1

    return {
        "llm_call_count": len(logs),
        "success_count": success_count,
        "failure_count": failure_count,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "runtime_seconds": runtime_seconds,
    }


def call_openai_chat(
    messages: List[Dict[str, str]],
    model: str,
    temperature: float,
    call_type: str,
    question: Optional[str] = None,
) -> Tuple[str, Dict[str, Any]]:
    """
    Backward-compatible function name.

    Although this function is still named call_openai_chat to avoid modifying
    other parts of the prototype, it now calls the DeepSeek Chat Completions API.
    """

    if not os.environ.get("DEEPSEEK_API_KEY"):
        raise EnvironmentError("Please set DEEPSEEK_API_KEY.")

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "top_p": 1.0,
        "n": 1,
        "stream": False,
        "presence_penalty": 0,
        "frequency_penalty": 0,
        "thinking": {
            "type": DEEPSEEK_THINKING_TYPE
        },
    }

    start_time = time.time()

    try:
        response = requests.post(
            DEEPSEEK_CHAT_COMPLETIONS_URL,
            headers=headers,
            json=payload,
            stream=False,
            timeout=120,
        )

        runtime_seconds = time.time() - start_time

        try:
            result = response.json()
        except Exception:
            result = {}

        if response.status_code != 200:
            log = {
                "call_type": call_type,
                "llm_provider": LLM_PROVIDER,
                "model": model,
                "model_display_name": LLM_DISPLAY_NAME,
                "api_base_url": DEEPSEEK_BASE_URL,
                "temperature": temperature,
                "deepseek_thinking_type": DEEPSEEK_THINKING_TYPE,
                "question": question,
                "runtime_seconds": runtime_seconds,
                "response_status_code": response.status_code,
                "usage": normalize_usage(result.get("usage") if isinstance(result, dict) else None),
                "success": False,
                "error_message": (
                    result.get("error", {}).get("message")
                    if isinstance(result, dict) and isinstance(result.get("error"), dict)
                    else response.text[:2000]
                ),
                "raw_response": response.text[:2000],
            }

            return "", log

        output = result["choices"][0]["message"].get("content", "")
        usage = normalize_usage(result.get("usage"))

        log = {
            "call_type": call_type,
            "llm_provider": LLM_PROVIDER,
            "model": model,
            "model_display_name": LLM_DISPLAY_NAME,
            "api_base_url": DEEPSEEK_BASE_URL,
            "temperature": temperature,
            "deepseek_thinking_type": DEEPSEEK_THINKING_TYPE,
            "question": question,
            "runtime_seconds": runtime_seconds,
            "response_status_code": response.status_code,
            "usage": usage,
            "success": True,
            "error_message": None,
            "has_reasoning_content": bool(result["choices"][0]["message"].get("reasoning_content")),
        }

        return output, log

    except Exception as exc:
        runtime_seconds = time.time() - start_time

        log = {
            "call_type": call_type,
            "llm_provider": LLM_PROVIDER,
            "model": model,
            "model_display_name": LLM_DISPLAY_NAME,
            "api_base_url": DEEPSEEK_BASE_URL,
            "temperature": temperature,
            "deepseek_thinking_type": DEEPSEEK_THINKING_TYPE,
            "question": question,
            "runtime_seconds": runtime_seconds,
            "response_status_code": None,
            "usage": normalize_usage(None),
            "success": False,
            "error_message": str(exc),
        }

        return "", log


# =========================================================
# 11. Question matching and answer refinement
# =========================================================

def load_question_bank_for_task(task_type: str) -> str:
    if task_type == "regression":
        return loader.load_regression_questions()
    return loader.load_classifier_questions()


def question_matching(
    question: str,
    question_bank_content: str,
    messages: List[Dict[str, str]],
) -> Tuple[int, str, Dict[str, Any], List[Dict[str, str]]]:
    query = (
        "My question is: " + question + "\n"
        "Please refer to the following question bank and choose the Section number "
        "(for example, if you choose Section 5, please return 5.) that matches the meaning of my question. "
        "Please note that as long as the meaning matches, there is no need for word-for-word correspondence. "
        "My entry may have spelling or grammatical mistakes, please ignore those mistakes. "
        "Returns 0 if no section matches. Only answer an integer as you choose, do not reply with any information "
        "other than the integer, do not reply why you chose the section number, do not reply to your thought process. "
        "Following is the question bank: \n" + question_bank_content
    )

    call_messages = messages + [{"role": "user", "content": query}]

    output, log = call_openai_chat(
        messages=call_messages,
        model=OPENAI_MODEL,
        temperature=MATCHING_TEMPERATURE,
        call_type="question_matching",
        question=question,
    )

    section_number = extract_first_integer(output)

    updated_messages = messages + [
        {"role": "user", "content": query},
        {"role": "assistant", "content": output},
    ]

    return section_number, output, log, updated_messages


def answer_refinement(
    default_answer: str,
    messages: List[Dict[str, str]],
    question: str,
) -> Tuple[str, Dict[str, Any], List[Dict[str, str]]]:
    call_messages = messages + [{"role": "user", "content": default_answer}]

    output, log = call_openai_chat(
        messages=call_messages,
        model=OPENAI_MODEL,
        temperature=REFINEMENT_TEMPERATURE,
        call_type="answer_refinement",
        question=question,
    )

    updated_messages = messages + [
        {"role": "user", "content": default_answer},
        {"role": "assistant", "content": output},
    ]

    return output, log, updated_messages


# =========================================================
# 12. Template Q&A generation
# =========================================================

def generate_template_qa(
    section_number: int,
    config: Dict[str, Any],
    fit_results: Dict[str, Any],
) -> Tuple[Any, Any]:
    task_type = fit_results["task_type"]
    y_column = config["y_column"]

    if task_type == "regression":
        coef_df = fit_results.get("coeff_df")

        if section_number == 1:
            return nlg_reg.Q_and_A_about_R2(
                coef_df,
                y_column,
                fit_results.get("selected_model"),
                fit_results.get("r_squared"),
            )

        if section_number == 2:
            return nlg_reg.Q_and_A_about_coefficients(
                coef_df,
                y_column,
                numeric_standardized=fit_results.get("numeric_standardized", False),
            )

        if section_number == 3:
            return nlg_reg.Q_and_A_about_importance(
                fit_results.get("preprocessed_feature_names"),
                y_column,
                fit_results.get("coeff"),
                fit_results.get("p_values"),
            )

        if section_number == 4:
            return nlg_reg.Q_and_A_about_pvalues(
                coef_df,
                y_column,
            )

        if section_number == 5:
            return nlg_reg.Q_and_A_about_ML_importance(
                fit_results.get("preprocessed_feature_names"),
                y_column,
                coef_df,
            )

        if section_number == 6:
            return nlg_reg.Q_and_A_about_ML_overfit(
                fit_results.get("train_r2"),
                y_column,
                fit_results.get("test_r2"),
            )

        return [], []

    coeff_df = fit_results.get("coeff_df")

    if section_number == 1:
        return nlg_cls.Q_and_A_about_accuracy(
            fit_results.get("accuracy"),
            fit_results.get("selected_model"),
        )

    if section_number == 2:
        return nlg_cls.Q_and_A_about_coefficients(
            coeff_df,
            y_column,
        )

    if section_number == 3:
        model_obj = fit_results.get("model")
        target_classes = getattr(model_obj, "classes_", [])

        return nlg_cls.Q_and_A_about_importance(
            fit_results.get("coeff"),
            y_column,
            coeff_df,
            target_classes,
        )

    if section_number == 4:
        return nlg_cls.Q_and_A_about_ML_overfit(
            fit_results.get("train_accuracy"),
            fit_results.get("test_accuracy"),
        )

    if section_number == 5:
        return nlg_cls.Q_and_A_about_ML_importance(
            coeff_df,
            y_column,
        )

    return [], []


# =========================================================
# 13. Run one question and one dataset
# =========================================================

def run_one_question(
    config: Dict[str, Any],
    fit_results: Dict[str, Any],
    question: str,
) -> Dict[str, Any]:
    messages = [
        {
            "role": "system",
            "content": config["background_knowledge"],
        }
    ]

    question_logs = []

    task_type = fit_results["task_type"]
    question_bank_content = load_question_bank_for_task(task_type)

    section_number, matching_raw_output, matching_log, messages = question_matching(
        question=question,
        question_bank_content=question_bank_content,
        messages=messages,
    )
    question_logs.append(matching_log)

    template_questions = None
    template_answers = None
    default_answer = None

    try:
        template_questions, template_answers = generate_template_qa(
            section_number=section_number,
            config=config,
            fit_results=fit_results,
        )

        default_answer = set_for_GPT.answer_update(
            question,
            template_questions,
            template_answers,
        )

        preprocessing_note = (
            fit_results.get("coefficient_interpretation_note")
            or fit_results.get("preprocessing_mode")
            or fit_results.get("analysis_preprocessing_mode")
        )

        if preprocessing_note:
            default_answer = (
                str(default_answer)
                + "\n\nImportant preprocessing and coefficient interpretation note:\n"
                + str(preprocessing_note)
            )

        final_answer, refinement_log, messages = answer_refinement(
            default_answer=default_answer,
            messages=messages,
            question=question,
        )
        question_logs.append(refinement_log)

        if not final_answer:
            status = "failure"
            failure_reason = "DeepSeek refinement returned an empty answer."
        else:
            status = "success"
            failure_reason = None

    except Exception as exc:
        final_answer = None
        status = "failure"
        failure_reason = str(exc)

        print(f"Template/refinement failed for question: {question}")
        print(f"Matched section: {section_number}")
        print(f"Failure reason: {failure_reason}")

    output = {
        "question": question,
        "matched_section": section_number,
        "matching_raw_output": matching_raw_output,
        "template_questions": sanitize_for_json(template_questions),
        "template_answers": sanitize_for_json(template_answers),
        "default_answer": default_answer,
        "final_answer": final_answer,
        "status": status,
        "failure_reason": failure_reason,
        "llm_logs": question_logs,
        "summary": summarize_llm_logs(question_logs),
    }

    return output


def summarize_dataset_question_logs(question_outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    all_logs = []
    success_count = 0
    failure_count = 0

    for q_out in question_outputs:
        all_logs.extend(q_out.get("llm_logs", []))

        if q_out.get("status") == "success":
            success_count += 1
        else:
            failure_count += 1

    summary = summarize_llm_logs(all_logs)

    summary.update(
        {
            "num_questions": len(question_outputs),
            "question_success_count": success_count,
            "question_failure_count": failure_count,
            "question_failure_rate": failure_count / len(question_outputs) if question_outputs else None,
        }
    )

    return summary


def run_one_dataset(config: Dict[str, Any]) -> Dict[str, Any]:
    dataset_name = config["dataset_name"]
    model_name = get_model_name(config)
    task_type = infer_task_type(config)

    output = {
        "baseline_name": "qckur_prototype_batch",
        "llm_provider": LLM_PROVIDER,
        "model_display_name": LLM_DISPLAY_NAME,
        "model_id": LLM_MODEL_ID,
        "api_base_url": DEEPSEEK_BASE_URL,
        "deepseek_thinking_type": DEEPSEEK_THINKING_TYPE,
        "matching_temperature": MATCHING_TEMPERATURE,
        "refinement_temperature": REFINEMENT_TEMPERATURE,
        "cost_measurement_mode": "per_question_isolated_messages",
        "analysis_preprocessing_mode": "unified_sklearn_pipeline_simpleimputer_standardscaler_onehotencoder",
        "dataset_name": dataset_name,
        "file_name": config.get("file_name"),
        "dataset_path": get_dataset_path(config),
        "background_knowledge": config["background_knowledge"],
        "x_columns": config["x_columns"],
        "y_column": config["y_column"],
        "task_type": task_type,
        "configured_model_name": model_name,
        "questions_and_answers": [],
    }

    try:
        fit_results = fit_model_with_unified_backend(config)

        output["fit_status"] = "success"
        output["fit_failure_reason"] = None
        output["fit_results"] = sanitize_for_json(fit_results)

        if task_type == "regression":
            output["numeric_standardized"] = fit_results.get("numeric_standardized")
            output["coefficient_interpretation_mode"] = fit_results.get("coefficient_interpretation_mode")
            output["coefficient_interpretation_note"] = fit_results.get("coefficient_interpretation_note")

    except Exception as exc:
        output["fit_status"] = "failure"
        output["fit_failure_reason"] = str(exc)
        output["fit_results"] = None

        for question in config["questions"]:
            output["questions_and_answers"].append(
                {
                    "question": question,
                    "status": "failure",
                    "failure_reason": f"Model fitting failed: {str(exc)}",
                    "final_answer": None,
                    "llm_logs": [],
                    "summary": summarize_llm_logs([]),
                }
            )

        output["dataset_summary"] = summarize_dataset_question_logs(output["questions_and_answers"])
        return output

    for question in config["questions"]:
        try:
            q_output = run_one_question(
                config=config,
                fit_results=fit_results,
                question=question,
            )

        except Exception as exc:
            q_output = {
                "question": question,
                "status": "failure",
                "failure_reason": str(exc),
                "final_answer": None,
                "llm_logs": [],
                "summary": summarize_llm_logs([]),
            }

        output["questions_and_answers"].append(q_output)

    output["dataset_summary"] = summarize_dataset_question_logs(output["questions_and_answers"])

    return output


# =========================================================
# 14. Batch run and summaries
# =========================================================

def summarize_all_dataset_outputs(outputs: List[Dict[str, Any]]) -> Dict[str, Any]:
    total_questions = 0
    total_question_failures = 0
    total_llm_calls = 0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_tokens = 0
    total_runtime_seconds = 0.0

    for output in outputs:
        s = output.get("dataset_summary", {}) or {}

        total_questions += s.get("num_questions", 0) or 0
        total_question_failures += s.get("question_failure_count", 0) or 0
        total_llm_calls += s.get("llm_call_count", 0) or 0
        total_prompt_tokens += s.get("prompt_tokens", 0) or 0
        total_completion_tokens += s.get("completion_tokens", 0) or 0
        total_tokens += s.get("total_tokens", 0) or 0
        total_runtime_seconds += s.get("runtime_seconds", 0.0) or 0.0

    return {
        "num_datasets": len(outputs),
        "total_questions": total_questions,
        "total_question_failures": total_question_failures,
        "overall_question_failure_rate": total_question_failures / total_questions if total_questions else None,
        "total_llm_call_count": total_llm_calls,
        "total_prompt_tokens": total_prompt_tokens,
        "total_completion_tokens": total_completion_tokens,
        "total_tokens": total_tokens,
        "total_runtime_seconds": total_runtime_seconds,
        "avg_llm_call_count_per_answer": total_llm_calls / total_questions if total_questions else None,
        "avg_total_tokens_per_answer": total_tokens / total_questions if total_questions else None,
        "avg_runtime_seconds_per_answer": total_runtime_seconds / total_questions if total_questions else None,
    }


def main():
    if not os.environ.get("DEEPSEEK_API_KEY"):
        raise EnvironmentError("Please set DEEPSEEK_API_KEY.")

    if os.environ.get("DEEPSEEK_API_KEY") == "your_deepseek_api_key_here":
        raise EnvironmentError("Please replace the placeholder DeepSeek API key before running.")

    configs = load_configs(CONFIG_PATH)
    all_outputs = []

    for config in configs:
        dataset_name = config["dataset_name"]
        model_name = get_model_name(config)

        print("=" * 100)
        print(f"Running QCKUR prototype dataset: {dataset_name}")
        print(f"Configured model: {model_name}")
        print(f"LLM provider: {LLM_PROVIDER}")
        print(f"LLM model: {LLM_MODEL_ID}")
        print("=" * 100)

        result = run_one_dataset(config)
        all_outputs.append(result)

        dataset_output_path = os.path.join(
            OUTPUT_DIR,
            f"{safe_name(dataset_name)}_{safe_name(model_name)}_prototype_deepseek_v4_outputs.json"
        )

        with open(dataset_output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print("Dataset summary:")
        print(json.dumps(result["dataset_summary"], ensure_ascii=False, indent=2))

        if result.get("task_type") == "regression":
            print("Numeric standardized:", result.get("numeric_standardized"))
            print("Coefficient interpretation mode:", result.get("coefficient_interpretation_mode"))

        for qa in result["questions_and_answers"]:
            print("-" * 80)
            print(qa["question"])
            print("Status:", qa["status"])
            print("Matched section:", qa.get("matched_section"))
            print("LLM calls:", qa["summary"]["llm_call_count"])
            print("Prompt tokens:", qa["summary"]["prompt_tokens"])
            print("Completion tokens:", qa["summary"]["completion_tokens"])
            print("Total tokens:", qa["summary"]["total_tokens"])
            print("Runtime seconds:", qa["summary"]["runtime_seconds"])
            print("Failure reason:", qa.get("failure_reason"))
            print("Final answer:")
            print(qa.get("final_answer"))
            print()

        print(f"Saved dataset output to: {dataset_output_path}")

    combined_output = {
        "baseline_name": "qckur_prototype_batch",
        "llm_provider": LLM_PROVIDER,
        "model_display_name": LLM_DISPLAY_NAME,
        "model_id": LLM_MODEL_ID,
        "api_base_url": DEEPSEEK_BASE_URL,
        "deepseek_thinking_type": DEEPSEEK_THINKING_TYPE,
        "matching_temperature": MATCHING_TEMPERATURE,
        "refinement_temperature": REFINEMENT_TEMPERATURE,
        "cost_measurement_mode": "per_question_isolated_messages",
        "analysis_preprocessing_mode": "unified_sklearn_pipeline_simpleimputer_standardscaler_onehotencoder",
        "config_path": CONFIG_PATH,
        "outputs": all_outputs,
        "overall_summary": summarize_all_dataset_outputs(all_outputs),
    }

    combined_output_path = os.path.join(
        OUTPUT_DIR,
        f"all_{safe_name(Path(CONFIG_PATH).stem)}_prototype_deepseek_v4_outputs.json"
    )

    with open(combined_output_path, "w", encoding="utf-8") as f:
        json.dump(combined_output, f, ensure_ascii=False, indent=2)

    print("=" * 100)
    print("Overall summary:")
    print(json.dumps(combined_output["overall_summary"], ensure_ascii=False, indent=2))
    print(f"Saved combined output to: {combined_output_path}")


main()